# Pragmatic naming experiment analysis

The accuracy of Naming Understandability Scores assigned by the pragmatic evaluator is compared with and without the Context Description.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from pymongo import MongoClient

EXPERIMENTS = {
    "pragmatic-context-gpt5-6-luna": "With context",
    "pragmatic-no-context-gpt5-6-luna": "Without context",
}
COLLECTION_NAME = "pragmatic_naming_eval"
MONGODB_URI = os.getenv(
    "MONGODB_URI",
    "mongodb://127.0.0.1:27017/llm_uml_evaluator?replicaSet=rs0",
)

plt.style.use("seaborn-v0_8-whitegrid")

## Data loading

The URI is read from the `MONGODB_URI` environment variable. JupyterLab is started with `uv run --env-file .env --group eval jupyter lab`.

In [ ]:
client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=5_000)
try:
    collection = client.get_default_database()[COLLECTION_NAME]
    documents = list(
        collection.find(
            {"experiment_name": {"$in": list(EXPERIMENTS)}},
            {
                "experiment_name": 1,
                "sample": 1,
                "mutation": 1,
                "nodes": 1,
            },
        )
    )
finally:
    client.close()

loaded_experiments = {document["experiment_name"] for document in documents}
if missing := set(EXPERIMENTS).difference(loaded_experiments):
    raise ValueError(f"No observations found for: {sorted(missing)}")

pd.Series(
    (document["experiment_name"] for document in documents),
    name="experiment_name",
).value_counts().rename("observations")

## Metric calculation

A classification pair contains a node UID and its Naming Understandability Score:

- `TP`: the returned score for a node is the expected score;
- `FP`: the returned node-score pair is not expected;
- `FN`: the expected node-score pair was not returned.

```text
Precision = TP / (TP + FP)
Recall    = TP / (TP + FN)
F1        = 2TP / (2TP + FP + FN)
```

Each node has one expected and one returned score, so a wrong score contributes one FP and one FN. Therefore `FP = FN`, precision, recall, and F1 are identical, and the formula simplifies to:

```text
F1 = TP / (TP + FN) = correctly scored nodes / all nodes
```

Only F1 is retained because the other two metrics carry the same information.

In [ ]:
df = pd.json_normalize(documents).rename(
    columns={
        "nodes.true_positive": "tp",
        "nodes.false_positive": "fp",
        "nodes.false_negative": "fn",
    }
)

required_columns = {
    "experiment_name", "sample", "mutation", "tp", "fp", "fn"
}
if missing := required_columns.difference(df.columns):
    raise ValueError(f"Missing columns: {sorted(missing)}")

df["context"] = df["experiment_name"].map(EXPERIMENTS)
f1_denominator = 2 * df["tp"] + df["fp"] + df["fn"]
df["f1"] = 2 * df["tp"] / f1_denominator.where(
    f1_denominator.ne(0)
)
df["all_nodes_correct"] = df["fp"].eq(0) & df["fn"].eq(0)

df[["experiment_name", "sample", "mutation", "context", "f1", "all_nodes_correct"]].head()

## Coverage check

The table shows the number of repetitions for each `experiment × sample × mutation` combination.

In [ ]:
coverage = (
    df.groupby(["context", "sample", "mutation"])
    .size()
    .unstack(fill_value=0)
)
display(coverage)
display(
    df.groupby(["context", "mutation"])["f1"]
    .agg(["mean", "std", "count"])
    .round(3)
)

## Context impact by mutation

The difference is calculated as `With context − Without context`; a positive value indicates an improvement from the Context Description.

In [ ]:
f1_comparison = df.groupby(["mutation", "context"])["f1"].mean().unstack()
all_correct_comparison = (
    df.groupby(["mutation", "context"])["all_nodes_correct"]
    .mean()
    .unstack()
)
comparison = pd.concat(
    {"F1": f1_comparison, "All nodes correct rate": all_correct_comparison},
    axis=1,
)
for metric in ("F1", "All nodes correct rate"):
    comparison[(metric, "Difference")] = (
        comparison[(metric, "With context")]
        - comparison[(metric, "Without context")]
    )
comparison = comparison.reindex(
    columns=pd.MultiIndex.from_product(
        [
            ("F1", "All nodes correct rate"),
            ("With context", "Without context", "Difference"),
        ]
    )
)
comparison.style.format(
    {
        ("F1", column): "{:.3f}"
        for column in ("With context", "Without context", "Difference")
    }
    | {
        ("All nodes correct rate", column): "{:.1%}"
        for column in ("With context", "Without context", "Difference")
    }
)

## Quality by mutation

Points show mean F1 for each context mode; vertical bars show one standard deviation across observations.

In [ ]:
f1_by_mutation = df.groupby(["context", "mutation"])["f1"].agg(
    ["mean", "std"]
)

fig, ax = plt.subplots(figsize=(10, 5))
for context in EXPERIMENTS.values():
    values = f1_by_mutation.loc[context]
    ax.errorbar(
        values.index,
        values["mean"],
        yerr=values["std"],
        marker="o",
        capsize=4,
        label=context,
    )
ax.set(
    title="Pragmatic naming F1 by mutation",
    xlabel="Mutation",
    ylabel="Score",
    ylim=(0, 1.05),
)
ax.legend()
plt.show()

## All nodes correct rate

An observation counts as correct only when every evaluated node receives the expected Naming Understandability Score.

In [ ]:
all_correct_by_mutation = (
    df.groupby(["mutation", "context"])["all_nodes_correct"]
    .mean()
    .unstack("context")
    .reindex(columns=EXPERIMENTS.values())
)

fig, ax = plt.subplots(figsize=(9, 4))
all_correct_by_mutation.plot.bar(ax=ax, rot=0)
for container in ax.containers:
    ax.bar_label(
        container,
        labels=[f"{value:.0%}" for value in container.datavalues],
        padding=3,
    )
ax.set(
    title="All nodes correct by mutation",
    xlabel="Mutation",
    ylabel="All nodes correct rate",
    ylim=(0, 1.08),
)
plt.show()